In [ ]:
import sys
sys.path.append('../Bayesflow/')
import bayesflow as bf
import numpy as np
import numba
import pandas as pd
from numba import njit
import keras
import matplotlib.pyplot as plt
import seaborn as sns
import notebook
import pandas as pd
import pickle
from scipy.stats import skew

In [ ]:
from simulator import meta, prior_inc, ddm, ddm_inc, likelihood, likelihood_inc

# Start with simulators

In [ ]:
# this is for increasing drift as a function of time

In [ ]:
prior_simulator = bf.simulators.LambdaSimulator(prior_inc)
likelihood_simulator_inc = bf.simulators.LambdaSimulator(likelihood_inc)
simulator_inc = bf.simulators.make_simulator([prior_simulator, likelihood_simulator_inc], meta_fn = meta)

In [ ]:
# check simulator visually

In [ ]:
resp = simulator_inc.sample(10000)

In [ ]:
sns.histplot(resp['rts'].flatten(), bins=1000, kde=True)

In [ ]:
sns.histplot(resp['rts'].mean(axis=1), bins=1000, kde=True)

In [ ]:
sns.histplot(resp['rts'].std(axis=1), bins=1000, kde=True)

In [ ]:
# prior_samples = simulator.sample(500)
# f = bf.diagnostics.plots.pairs_samples(
#     variable_keys = param_keys,
#     variable_names = param_names
# )

In [ ]:
@keras.saving.register_keras_serializable(package="custom_package")
def forward_transform(n):
    return np.sqrt(n)

@keras.saving.register_keras_serializable(package="custom_package")
def inverse_transfrom(n):
    return n ** 2

In [ ]:
adapter = (
    bf.Adapter()
    .broadcast("n_obs", to="rts")
    .as_set(["rts", "timeouts"])
    .standardize(exclude=["n_obs"])
    .apply(include="n_obs", forward=forward_transform, inverse=inverse_transfrom)
    .convert_dtype(from_dtype="float64", to_dtype="float32")
    .concatenate(["dv", "a", "t0"], into="inference_variables")
    .concatenate(["rts","timeouts"], into="summary_variables")
    .rename("n_obs", "inference_conditions")
)

## Networks and Workflow

In [ ]:
summary_net = bf.networks.DeepSet(summary_dim=8, depth=2)
inference_net = bf.networks.CouplingFlow(depth=4)

In [ ]:
workflow = bf.BasicWorkflow(
    # simulator=simulator,
    simulator=simulator_inc, # increasing drift
    adapter=adapter,
    inference_network=inference_net,
    summary_network=summary_net,
    initial_learning_rate=1e-4, #5e-4,
    optimizer=keras.optimizers.AdamW,
    inference_variables=["dv", "a", "t0"],
    summary_variables=["rts", "timeouts"],
    inference_conditions=["n_obs"],
    checkpoint_filepath=f"checkpoints/newcomplex_ddm1",
    checkpoint_name="one_bound_ddm_complex1",    
)

In [ ]:
history = workflow.fit_online(epochs=250, 
                              batch_size=32)

In [ ]:
# for later reloading the checkpoints: 
# increasing drift
checkpoint_path=f"checkpoints/newcomplex_ddm1/one_bound_ddm_complex1.keras"
workflow.approximator = keras.saving.load_model(checkpoint_path)

## Diagnostics

In [ ]:
test_sims = workflow.simulate(300)

In [ ]:
workflow.plot_diagnostics(test_sims)

In [ ]:
param_names = ["Inc in drift $\Delta_v$", "Boundary $a$", "Non-decision time $t_0$"]
param_keys = ["$\Delta_v$", "a", "t0"]

In [ ]:
# f = bf.diagnostics.plots.loss(history)
# f.savefig("plots/loss.pdf")

In [ ]:
num_samples = 1000

test_sims1 = workflow.simulate(300)
data = test_sims1.pop("rts")
n_obs = test_sims1.pop("n_obs")
timeouts = test_sims1.pop("timeouts")

samples = workflow.sample(conditions={"rts":data, "n_obs":n_obs, "timeouts":timeouts}, num_samples=num_samples)

In [ ]:
f = bf.diagnostics.plots.calibration_ecdf(samples, 
                                          test_sims1,
                                          
                                          variable_names=param_names,
                                          difference=True,
                                          label_fontsize=14,
                                          # legend=False,
                                          legend_location="left", 
                                          legend_fontsize=10,
                                                                             
                                          rank_ecdf_color = '#000000',
                                          set_layout_engine=True,
                                          
                                          # rank_type='distance'
                                         )

f.savefig("plots/newmodel/calibration_ecdf_complex1.pdf")

In [ ]:
# f = bf.diagnostics.plots.calibration_histogram(samples, test_sims1)#, variable_names=param_names)
# f.savefig("plots/newmodel/calib_hist_complex.pdf")

In [ ]:
f = bf.diagnostics.plots.recovery(samples, test_sims1, 
                                  variable_names=param_names,
                                  label_fontsize=14,
                                  color='#000000')
f.savefig("plots/newmodel/plot_recovery_complex.pdf")

# Simulated RT check

In [ ]:
prior_samples = simulator_inc.sample(10000)

In [ ]:
sns.kdeplot(prior_samples["rts"].mean(axis=1), fill=True)

In [ ]:
sns.kdeplot(prior_samples["rts"].std(axis=1), fill=True)

In [ ]:
sns.kdeplot(skew(prior_samples["rts"], axis=1), fill=True)

## Model checking

### 1. Posterior retrodictive Check

In [ ]:
num_samples = 1000
num_sim = 10
num_resim = 50

sims = workflow.simulate(num_sim)
rts = sims.pop("rts")
n_obs = sims.pop("n_obs")
timeouts = sims.pop("timeouts")
# fit model and draw 1000 samples per dataset
post_samples = workflow.sample(conditions={"rts":rts, "n_obs":n_obs, "timeouts":timeouts}, num_samples=num_samples)

In [ ]:
post_samples.keys()

In [ ]:
index_set = np.random.choice(np.arange(num_samples), size=num_resim)

In [ ]:
# get mean values for each participant
post_samples_mean = {k: np.mean(t, axis=1) for k, t in post_samples.items()}

In [ ]:
dv = post_samples.pop('dv')
a = post_samples.pop('a')
t0 = post_samples.pop('t0')
post_samples_np = np.concatenate([dv, a, t0], axis=2)

In [ ]:
# Re-simulate
pred_data = np.zeros((num_sim, num_resim, n_obs, 1))

for sim in range(num_sim):
    # for i, idx in enumerate(index_set):
    for i, idx in enumerate(index_set):
        dv, a, t0 = post_samples_np[sim, idx, :]
        rt = ddm_inc(dv, a, t0, n_obs)
        pred_data[sim, i, :, :] = rt[:, 0][:, np.newaxis]
        # pred_data[sim, i, :, :] = ddm(post_samples_np[sim, idx], n_obs)

In [ ]:
# Check simulated RTs distributions
f, axarr = plt.subplots(2, 5, figsize=(12, 4))
for i, ax in enumerate(axarr.flat):
    sns.histplot(rts[i, :].flatten(), ax=ax)
    sns.despine(ax=ax)
    ax.set_label("")
    ax.set_yticks([])
    if i > 4:
        ax.set_xlabel("Sim RTs")
f.tight_layout()

In [ ]:
f, axarr = plt.subplots(2, 4, figsize=(18, 8))
for i, ax in enumerate(axarr.flat):
    sns.kdeplot(
        rts[i, :].flatten(), ax=ax, fill=True, color="black", alpha=0.5, label="sim_data")
    sns.kdeplot(
        pred_data[i, :, :, 0].flatten(), ax=ax, fill=True, color="maroon", alpha=0.5, label="pred_data")
    ax.set_xlim((0, 20))
    ax.set_ylabel("")
    ax.set_yticks([])
    ax.set_title(f"Simulated data set #{i+1}", fontsize=18)

    # Set legeng to first plot
    if i==0:
        ax.legend()
    if i > (num_sim//2)-1:
        ax.set_xlabel("Response time (secs)", fontsize=16)
    sns.despine()
f.tight_layout()

### 2. Posterior Predictive check

# READ THE DATA

In [ ]:
# Prepare subject data for model fitting

In [ ]:
# # DON'T RUN THIS AGAIN!!
# data = pd.read_csv("datasets/HC3_complex.csv")
# subs = data.RELEASEID.unique()
# #subs_small = subs[0:10]
# N_SUBS = len(subs)
# #N_SUBS = 10

# subject_dicts = {}

# for sub in subs:
#     person_data = data[data.RELEASEID == sub]
#     rt = np.array(person_data.TIME)[np.newaxis, :]
#     n_obs = rt.shape[1]
#     timeouts = np.zeros(n_obs)
    
#     subject_dicts[sub] = {"rts":rt,
#                           "n_obs":n_obs,
#                           "timeouts":timeouts}

# with open("datasets/HC3_complex_inc.pkl", "wb") as f:
#     pickle.dump(subject_dicts, f)

In [ ]:
# # DON'T RUN THIS AGAIN!!
# # prepate the data from HC5
# data = pd.read_csv("datasets/HC5_complex.csv")
# subs = data.RELEASEID.unique()
# #subs_small = subs[0:10]
# N_SUBS = len(subs)
# # N_SUBS = 10

# subject_dicts = {}

# for sub in subs:
#     person_data = data[data.RELEASEID == sub]
#     rt = np.array(person_data.TIME)[np.newaxis, :]
#     n_obs = rt.shape[1]
#     timeouts = np.zeros(n_obs)

#     subject_dicts[sub] = {"rts":rt,
#                           "n_obs":n_obs,
#                          "timeouts":timeouts}

# with open("datasets/HC5_complex.pkl", "wb") as f:
#     pickle.dump(subject_dicts, f)

In [ ]:
# the subject dictionary is saved, load it here:
# the one with increasing drift
with open("datasets/HC3_complex_inc.pkl", "rb") as t:
    subject_dicts = pickle.load(t)


In [ ]:
len(subject_dicts.keys())

In [ ]:
subs_small=list(subject_dicts.keys())[0:10]

In [ ]:
# get the posterior samples
num_samples = 1000
subs = list(subject_dicts.keys())
# subs_small = subject_dicts[0:10]

posteriors = {}
for sub in subs:
    n_obs = subject_dicts[sub]["n_obs"]
    data  = subject_dicts[sub]["rts"]
    timeouts = subject_dicts[sub]["timeouts"]
    post = workflow.sample(conditions={"rts":data, "n_obs":n_obs, "timeouts":timeouts}, num_samples=num_samples)
    posteriors[sub] = post

In [ ]:
# increasing drift rate
with open("datasets/postHC3_inc.pkl", "wb") as f:
    pickle.dump(posteriors, f)

In [ ]:
with open("datasets/postHC3_inc.pkl", "rb") as f:
    posteriors = pickle.load(f)

In [ ]:
# GET POSTERIOR SAMPLES' SUMMARIES
posterior_means = {}

for sub in posteriors.keys():
    posterior_means[sub] = {param: {"mean": np.mean(posteriors[sub][param]),
                                    "median":np.median(posteriors[sub][param]),
                                    "std" : np.std(posteriors[sub][param], ddof=1)}
                                    for param in posteriors[sub].keys()}

In [ ]:
with open("datasets/postHC3_inc_sum.pkl", "wb") as f:
    pickle.dump(posterior_means, f)

In [ ]:
with open("datasets/postHC3_inc_sum.pkl", "rb") as f:
    posterior_means = pickle.load(f)

In [ ]:
sum_dataHC3 = []
for sub, params in posterior_means.items():
    for param, stats in params.items():
        sum_dataHC3.append({
            "RELEASEID": sub,
            "param":param,
            "mean":stats["mean"],
            "median":stats["median"],
            "std":stats["std"]
        })
df = pd.DataFrame(sum_dataHC3)

In [ ]:
paramss = df['param'].unique()

In [ ]:
fig, axes = plt.subplots(1, len(paramss), figsize=(6 * len(paramss), 5))
for i, p in enumerate(paramss):
    ax=axes[i]
    subset = df[df['param'] == p]
    # sns.histplot(subset['mean'], color='lightblue', label='mean', kde=True, ax=ax, stat='density', bins=100)
    sns.histplot(subset['median'], color='lightcoral', label='median', kde=True, ax=ax, stat='density', bins=100)

    ax.set_title(f"hist of {p}")
    ax.set_xlabel("value")
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# save it as csv for further analysis
df.to_csv("datasets/postsumHC3_complex_inc.csv", index=False)

# Read demographics data

In [ ]:
demog_HC3 = pd.read_csv("demo_subj_HC3.csv")
# demog_HC3 = pd.read_csv("demo_age_group.csv")